In [59]:
import os
import numpy as np
import scipy.stats as stats
import torch
import copy
from torch import nn
from torch import optim
import torch.nn.functional as F
import torchvision
from torchvision import datasets
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

In [60]:
with_cuda = torch.cuda.is_available()
if with_cuda:
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

In [61]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"


In [62]:
v = torch.tensor([1,2,3,4,5])
v = v.to(device)
print(v.device)

cuda:0


# Text prediction

## The Penn-TreeBank (PTB) dataset

**Question 1**

Take a look at the files `ptb_test.txt`, `ptb_valid.txt`, `ptb_train.txt`. What are the differences with a real text? Why?

In [63]:
import os

# Obtenir le répertoire de travail actuel
current_directory = os.getcwd()
datasets_place = current_directory
print(f"Répertoire de travail actuel : {datasets_place}")

Répertoire de travail actuel : c:\Users\gabri\Desktop\to push


In [64]:
datasets_place = current_directory + "\\ptb"

**Question 2**

We want to perform text prediction. For that, we have two choices: 
1. character prediction;
2. word prediction.

Specify the format of the input and the output of our model: how should we encode them?

**Question 3**

We want to replace the characters/words by tokens. For that, we build the classes `Dictionary`, `CorpusWords` and `CorpusChars`.

The class `Dictionary` has attributes:
* a dictionary `word2idx`, which maps words (or characters) to tokens;
* a list `idx2word`, which maps tokens to words (or characters);
* a dictionary `nb_occ`, which contains the number of occurrences of each word (or character).

Write the method `add_word`, which takes a word (or character), and updates the attributes accordingly.

The class `CorpusWords` has attributes:
* the dictionary of words `dictionary`:
* the tokenized train, valid and test texts.

Write the method `tokenize`, which takes a path to a text file and:
1. reads it sequentially word by word and add the words to the dictionary;
2. reads it sequentially word by word, transforms the words into token (by using the dictionary), and store the tokens into a LongTensor; each "new line" character should also be stored;
3. returns the LongTensor in which thetoken have been stored.

Build the class `CorpusChars` in the same way (by replacing words by characters).

In [65]:
class Dictionary:
    def __init__(self):
        self.word2idx = {}
        self.idx2word = []
        self.nb_occ = {}

    def add_word(self, word):
        if word not in self.word2idx.keys():
            if len(self.word2idx) == 0:
                cles_max = -1
            else:
                cles_max=len(self.idx2word)
            self.word2idx[word]= cles_max+1
            self.nb_occ[word] = 1
            self.idx2word.append(word)
        else:
            self.nb_occ[word]+=1
            cles_max = self.word2idx[word]
        return cles_max + 1
            
    def __len__(self):
        return len(self.idx2word)


class CorpusWords:
    def __init__(self, path):
        self.dictionary = Dictionary()
        self.train = self.tokenize(os.path.join(path, 'ptb_train.txt'))
        self.valid = self.tokenize(os.path.join(path, 'ptb_valid.txt'))
        self.test = self.tokenize(os.path.join(path, 'ptb_test.txt'))

    def tokenize(self, path):
        """Tokenizes a text file."""
        liste=[]
        with open(path, 'r') as f:
            for line in f:
                line=list(line.split())
                for word in line:
                    cles_max = self.dictionary.add_word(word)
                    liste.append(cles_max)
                    
        tenseur = torch.LongTensor(liste)
        return tenseur
  
                
                    
class CorpusChars:
    def __init__(self, path):
        self.dictionary = Dictionary()
        self.train = self.tokenize(os.path.join(path, 'ptb_train.txt'))
        self.valid = self.tokenize(os.path.join(path, 'ptb_valid.txt'))
        self.test = self.tokenize(os.path.join(path, 'ptb_test.txt'))

    def tokenize(self, path):
        """Tokenizes a text file."""
        liste=[]
        with open(path, 'r') as f:
            for line in f:
                line=list(line.split())
                for word in line:
                    for car in word:
                        cles_max = self.dictionary.add_word(car)
                        liste.append(cles_max)
                    
        tenseur = torch.LongTensor(liste)
        return tenseur
   

In [66]:
class Dictionary:
    def __init__(self):
        self.word2idx = {}
        self.idx2word = []
        self.nb_occ = {}

    def add_word(self, word):
        if word not in self.word2idx:
            self.word2idx[word] = len(self.idx2word)  # L'index commence à 0
            self.nb_occ[word] = 1
            self.idx2word.append(word)
        else:
            self.nb_occ[word] += 1
        return self.word2idx[word]  # Retourne l'index actuel (sans +1)
    
    def tokenize(self, path):
        """Tokenizes a text file."""
        liste = []
        with open(path, 'r') as f:
            for line in f:
                words = line.split()  # Split par défaut utilise l'espace comme séparateur
                for word in words:
                    idx = self.dictionary.add_word(word)
                    liste.append(idx)
    
        tenseur = torch.LongTensor(liste)
        return tenseur
    
    def __len__(self):
        return len(self.idx2word)

In [67]:
test_chars = CorpusChars(datasets_place)
test_mot = CorpusWords(datasets_place)


In [68]:
len(test_mot.dictionary)

9999

**Question 4** 

What does the following functions do?

In [69]:
def batchify(data, batch_size):
    nbatches = data.size(0) // batch_size
    data = data.narrow(0, 0, nbatches * batch_size)
    data = data.view(batch_size, -1).t().contiguous()
    
    return data

In [70]:
def get_batch(data_src, bptt, i):
    seq_len = min(bptt, len(data_src) - 1 - i)
    data = data_src[i:i + seq_len]
    target = data_src[i + 1:i + 1 + seq_len].view(-1)
    
    return data, target

## Building a LSTM

**Question 5**

Build a class `ModelRNN`:
* which takes arguments:
  * `ntokens`: the number of different tokens (= the size of the dictionary);
  * `nhid`: the number of neurons in the hidden layer of the LSTM;
  * `nlayers`: the number of stacked LSTMs;
* which has attributes:
  * `rnn`: a `nn.LSTM` initialized with the relevant attributes;
  * exercise: other attributes the are necessary to make it work;
* which implements the method `forward`:
  * exercise: implement it; what should be the arguments and the return values of this function?

In [71]:
class ModelRNN(nn.Module):
    def __init__(self, ntokens, ninp, nhid, nlayers):
        super(ModelRNN, self).__init__()
        self.encoder = nn.Embedding(ntokens,ninp)
        self.rnn = nn.LSTM(ninp, nhid, nlayers)
        self.decoder = nn.Linear(nhid,ntokens)
        self.nlayers = nlayers
        self.nhid = nhid
        self.ntokens = ntokens

    def forward(self, input, hidden, current):
        embedding = self.encoder(input)
        """print(embedding.size(),hidden.size(),current.size())
        print(len(list([embedding,hidden,current])))"""
        output, (hidden, current) = self.rnn(embedding, (hidden, current))
        output = output.view(-1,output.size(-1))
        output = self.decoder(output)
        output = F.log_softmax(output,dim=1)
        return output, (hidden,current)
        
    
    def init_hidden(self, batch_size):
        device = next(self.parameters()).device
        return (torch.zeros((self.nlayers, batch_size, self.nhid), device = device),
                torch.zeros((self.nlayers, batch_size, self.nhid), device = device))

**Question 6**

Complete the code in `train_model`

In [82]:
from tqdm import tqdm
def train_model(train_data, bptt, model, criterion, optimizer, nepochs, batch_size):
    train_losses = []
    train_acc = []

    for epoch in range(nepochs):
        train_loss = 0.
        correct = 0
        total_pred = 0

        model.train()
        taille = len(train_data)
        current, hidden = model.init_hidden(batch_size)

        batches = list(range(0, taille - 1, bptt))
        pbar = tqdm(enumerate(batches), total=len(batches),
                    desc=f"Epoch {epoch+1}/{nepochs}", leave=True)

        for batch_idx, i in pbar:
            data, targets = get_batch(train_data, bptt, i)
            data, targets = data.to(device), targets.to(device)

            optimizer.zero_grad()
            output, (hidden, current) = model(data, hidden, current)

            loss = criterion(output, targets)
            hidden.detach_()
            current.detach_()

            loss.backward()
            optimizer.step()

            train_loss += loss.item() * data.size(0)
            pred = output.data.max(1, keepdim=True)[1]
            correct += pred.eq(targets.data.view_as(pred)).sum().item()

            # Mise à jour de la barre en temps réel
            pbar.set_postfix({
                "loss": f"{loss.item():.4f}",
                "acc":  f"{correct / ((batch_idx + 1) * bptt * batch_size) * 100:.2f}%"
            })

        train_loss = train_loss / taille
        train_losses.append(train_loss)
        print(f"Epoch {epoch+1}/{nepochs} | Loss: {train_loss:.6f}")

**Question 7**

Construct the batches for character prediction and train the model.

In [73]:
v = torch.tensor([1,2,3,4,5])
v = v.to(device)
print(v.device)

cuda:0


In [75]:
batch_size = 10
bptt = 35

ninp = 200
nhid = 200
nlayers = 2

corpus = CorpusWords(datasets_place)
train_data = corpus.train
test_loader = batchify(train_data, batch_size)
ntokens = len(corpus.dictionary)  
print("Nombre de tokens uniques (vocab size):", ntokens)

model = ModelRNN(ntokens, ninp, nhid, nlayers).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.01)
nepochs = 20

train_model(test_loader, bptt, model, criterion, optimizer, nepochs , batch_size=batch_size )


Nombre de tokens uniques (vocab size): 9999
Epoch: 0 	Training Loss: 0.002293
Epoch: 1 	Training Loss: 0.002086
Epoch: 2 	Training Loss: 0.002006
Epoch: 3 	Training Loss: 0.001994
Epoch: 4 	Training Loss: 0.001932
Epoch: 5 	Training Loss: 0.001906
Epoch: 6 	Training Loss: 0.001886
Epoch: 7 	Training Loss: 0.001867
Epoch: 8 	Training Loss: 0.001851
Epoch: 9 	Training Loss: 0.001842
Epoch: 10 	Training Loss: 0.001834
Epoch: 11 	Training Loss: 0.001826
Epoch: 12 	Training Loss: 0.001822
Epoch: 13 	Training Loss: 0.001813
Epoch: 14 	Training Loss: 0.001809
Epoch: 15 	Training Loss: 0.001801
Epoch: 16 	Training Loss: 0.001797
Epoch: 17 	Training Loss: 0.001795
Epoch: 18 	Training Loss: 0.001791
Epoch: 19 	Training Loss: 0.001788


In [ ]:
def generate(model, corpus, prompt, n_tokens=50, temperature=1.0, mode="words"):
    model.eval()
    dico = corpus.dictionary

    if mode == "words":
        tokens = prompt.lower().strip().split()
    else:
        tokens = list(prompt.lower().strip().replace(" ", ""))

    # Remplace les tokens inconnus
    unknown = [t for t in tokens if t not in dico.word2idx]
    if unknown:
        tokens = [t if t in dico.word2idx else "<unk>" for t in tokens]

    # Encode
    ids = torch.LongTensor([dico.word2idx[t] for t in tokens]).to(device)

    with torch.no_grad():
        hidden, current = model.init_hidden(1)

        for i in range(len(ids) - 1):
            tok = ids[i].view(1, 1)
            _, (hidden, current) = model(tok, hidden, current)

        input_tok = ids[-1].view(1, 1)
        generated = list(tokens)

        for _ in range(n_tokens):
            output, (hidden, current) = model(input_tok, hidden, current)

            logits  = output[-1] / temperature
            probs   = torch.exp(logits)
            next_id = torch.multinomial(probs, 1).item()

            next_tok = dico.idx2word[next_id]
            generated.append(next_tok)
            input_tok = torch.LongTensor([[next_id]]).to(device).view(1, 1)

    sep = " " if mode == "words" else ""
    prompt_str    = sep.join(tokens)
    generated_str = sep.join(generated[len(tokens):])

    print(f"\n  PROMPT    : {prompt_str}")
    print(f"  GENERATED : {prompt_str} \033[92m{generated_str}\033[0m")
    return generated

In [77]:
prompts = [
        "the company said",
        "the stock market",
        "president bush",
        "new york",
    ]

for p in prompts:
    generate(model, corpus, prompt=p, n_tokens=10, temperature=0.8, mode="words")


  PROMPT    : the company said
  GENERATED : the company said he had looked for diamonds and other <unk> in the

  PROMPT    : the stock market
  GENERATED : the stock market crash is getting strongly the prime news they 're going

  PROMPT    : president bush
  GENERATED : president bush said <unk> de <unk> <unk> <unk> <unk> a steady to

  PROMPT    : new york
  GENERATED : new york are that they 's deputies indeed everything 's the <unk>


In [ ]:
print(" INTERACTIVE MODE  (tape 'quit' pour quitter)")
while True:
    prompt = input("\n Write the beginning of a sentence : ").strip()
    if prompt.lower() == "quit":
        break
    n = input("How many words to generate ? [default: 10] : ").strip()
    n = int(n) if n.isdigit() else 10
    t = input("Temperature ? [default: 1.0] : ").strip()
    t = float(t) if t else 1.0
    generate(model, corpus, prompt=prompt, n_tokens=n, temperature=t, mode="words")


  MODE INTERACTIF (tape 'quit' pour quitter)

  PROMPT    : i love
  GENERATED : i love has a credibility that among the government 's members nothing

  PROMPT    : the company employed
  GENERATED : the company employed a N N hastings but what it would be increased


In [83]:
batch_size = 10
bptt = 35

ninp = 200
nhid = 200
nlayers = 2

corpus_carac = CorpusChars(datasets_place)
train_data = corpus_carac.train
test_loader = batchify(train_data, batch_size)
ntokens = len(corpus_carac.dictionary)  
print("Nombre de tokens uniques (vocab size):", ntokens)

model2 = ModelRNN(ntokens, ninp, nhid, nlayers).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model2.parameters(), lr = 0.01)
nepochs = 5

train_model(test_loader, bptt, model2, criterion, optimizer, nepochs , batch_size=batch_size )


Nombre de tokens uniques (vocab size): 48


Epoch 1/5: 100%|██████████| 11800/11800 [03:37<00:00, 54.23it/s, loss=1.6915, acc=47.90%]


Epoch 1/5 | Loss: 1.756100


Epoch 2/5: 100%|██████████| 11800/11800 [03:10<00:00, 61.85it/s, loss=1.7263, acc=50.43%]


Epoch 2/5 | Loss: 1.664932


...

Epoch 3/5 | Loss: 1.674996


Epoch 4/5: 100%|██████████| 11800/11800 [03:10<00:00, 62.06it/s, loss=1.8058, acc=49.62%]


Epoch 4/5 | Loss: 1.692433


Epoch 5/5: 100%|██████████| 11800/11800 [03:10<00:00, 62.03it/s, loss=1.8761, acc=47.40%]

Epoch 5/5 | Loss: 1.770231


In [85]:
prompts = [
        "the company said",
        "the stock market",
        "president bush",
        "new york",
    ]

for p in prompts:
    generate(model2, corpus, prompt=p, n_tokens=50, temperature=0.8, mode="tokens")

⚠️  Tokens inconnus : ['t', 'h', 'o', 'm', 'p', 'n', 'y'] → remplacés par <unk>


RuntimeError: CUDA error: device-side assert triggered
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


**Question 8**

Do the same for word prediction. What is the purpose of the `encoder` and the `decoder` modules?